In [3]:
pip install lightgbm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 20.1 MB/s eta 0:00:0000:0100:01

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install lightgbm scikit-learn pandas numpy joblib


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [10]:
!brew install libomp

==> Auto-updating Homebrew...
Adjust how often this is run with `$HOMEBREW_AUTO_UPDATE_SECS` or disable with
`$HOMEBREW_NO_AUTO_UPDATE=1`. Hide these hints with `$HOMEBREW_NO_ENV_HINTS=1` (see `man brew`).
==> Downloading https://ghcr.io/v2/homebrew/core/portable-ruby/blobs/sha256:f41c72b891c40623f9d5cd2135f58a1b8a5c014ae04149888289409316276c72
######################################################################### 100.0%
==> Pouring portable-ruby-4.0.2_1.arm64_big_sur.bottle.tar.gz
==> Auto-updated Homebrew!
Updated 2 taps (homebrew/core and homebrew/cask).
==> New Formulae
apache-arrow-adbc: Cross-language, Arrow-native database access
apkeep: Command-line tool for downloading APK files from various sources
atuin-server: Sync server for atuin - Improved shell history for zsh, bash, fish and nushell
betterleaks: Secrets scanner built for configurability and speed
buildkitd: Concurrent, cache-efficient, and Dockerfile-agnostic builder toolkit (Daemon)
checkpwn: Check Have I Been Pwne

In [12]:
import lightgbm
print(lightgbm.__version__)

4.6.0


In [5]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder


RANDOM_STATE = 42
DATA_PATH = Path("/Users/apple/Documents/cfa/IT5006/models/final_crime_dataset_with_temporal.csv")
OUTPUT_DIR = Path("/Users/apple/Documents/cfa/IT5006/models")

TARGET_COL = "crime_category"
BASE_FEATURES = [
    "hour",
    "day_of_week",
    "month",
    "is_weekend",
    "community_area_id",
    "distance_to_nearest_station",
    "stations_within_500m",
    "community_type",
]

NUMERIC_FEATURES = [
    "hour",
    "day_of_week",
    "month",
    "is_weekend",
    "community_area_id",
    "distance_to_nearest_station",
    "stations_within_500m",
]

CATEGORICAL_FEATURES = ["community_type"]


def load_dataset(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(
            f"Dataset not found at {path.resolve()}. Place final_crime_dataset_with_temporal.csv next to this script or update DATA_PATH."
        )

    df = pd.read_csv(path)

    required_cols = [TARGET_COL] + BASE_FEATURES
    missing = [col for col in required_cols if col not in df.columns]
    if missing:
        raise ValueError(f"Dataset is missing required columns: {missing}")

    df = df[required_cols].dropna().copy()

    # Ensure stable dtypes for inference
    int_like_cols = ["hour", "day_of_week", "month", "is_weekend", "community_area_id", "stations_within_500m"]
    float_like_cols = ["distance_to_nearest_station"]

    for col in int_like_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    for col in float_like_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df["community_type"] = df["community_type"].astype(str).str.strip()
    df = df.dropna().copy()

    for col in int_like_cols:
        df[col] = df[col].astype(int)
    for col in float_like_cols:
        df[col] = df[col].astype(float)

    return df


def build_pipeline() -> Pipeline:
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "cat",
                OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                CATEGORICAL_FEATURES,
            ),
            ("num", "passthrough", NUMERIC_FEATURES),
        ],
        remainder="drop",
    )

    model = LGBMClassifier(
        objective="multiclass",
        class_weight="balanced",
        n_estimators=200,
        learning_rate=0.08,
        num_leaves=31,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=RANDOM_STATE,
        verbose=-1,
    )

    return Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", model),
    ])


def compute_feature_stats(df: pd.DataFrame) -> dict:
    stats = {}
    for col in NUMERIC_FEATURES:
        stats[col] = {
            "mean": float(df[col].mean()),
            "std": float(df[col].std(ddof=0)) if float(df[col].std(ddof=0)) > 0 else 0.0,
            "min": float(df[col].min()),
            "max": float(df[col].max()),
        }
    return stats


def top_k_preview(prob_row: np.ndarray, class_names: list[str], k: int = 3) -> list[dict]:
    idx = np.argsort(prob_row)[::-1][:k]
    return [
        {"label": class_names[i], "probability": float(prob_row[i])}
        for i in idx
    ]


def main() -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    df = load_dataset(DATA_PATH)
    X = df[BASE_FEATURES].copy()
    y = df[TARGET_COL].copy()

    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y_encoded,
        test_size=0.2,
        random_state=RANDOM_STATE,
        stratify=y_encoded,
    )

    pipeline = build_pipeline()
    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    report = classification_report(
        y_test,
        y_pred,
        target_names=label_encoder.classes_,
        output_dict=True,
        zero_division=0,
    )

    pipeline_path = OUTPUT_DIR / "chicago_crime_pipeline.pkl"
    labels_path = OUTPUT_DIR / "label_classes.json"
    metadata_path = OUTPUT_DIR / "model_metadata.json"
    feature_stats_path = OUTPUT_DIR / "feature_stats.json"
    sample_payloads_path = OUTPUT_DIR / "sample_payloads.json"

    joblib.dump(pipeline, pipeline_path)

    with open(labels_path, "w", encoding="utf-8") as f:
        json.dump(list(label_encoder.classes_), f, indent=2, ensure_ascii=False)

    metadata = {
        "model_type": "LightGBM multiclass pipeline",
        "target_column": TARGET_COL,
        "feature_columns": BASE_FEATURES,
        "numeric_features": NUMERIC_FEATURES,
        "categorical_features": CATEGORICAL_FEATURES,
        "random_state": RANDOM_STATE,
        "n_classes": int(len(label_encoder.classes_)),
        "class_labels": list(label_encoder.classes_),
        "metrics": {
            "accuracy": float(accuracy),
            "macro_f1": float(report.get("macro avg", {}).get("f1-score", 0.0)),
            "weighted_f1": float(report.get("weighted avg", {}).get("f1-score", 0.0)),
        },
        "notes": [
            "This deployment artifact is reconstructed from the provided notebooks.",
            "It uses the stable base feature set for inference.",
            "Any more complex engineered features from exploratory notebook cells are intentionally excluded for deployment simplicity.",
        ],
    }

    with open(metadata_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)

    with open(feature_stats_path, "w", encoding="utf-8") as f:
        json.dump(compute_feature_stats(df), f, indent=2, ensure_ascii=False)

    sample_payloads = []
    sample_rows = X_test.head(3).copy()
    sample_probs = pipeline.predict_proba(sample_rows)
    for i, (_, row) in enumerate(sample_rows.iterrows()):
        payload = row.to_dict()
        payload["top_3_predictions"] = top_k_preview(sample_probs[i], list(label_encoder.classes_), k=3)
        sample_payloads.append(payload)

    with open(sample_payloads_path, "w", encoding="utf-8") as f:
        json.dump(sample_payloads, f, indent=2, ensure_ascii=False)

    print("Saved:")
    print(f"  - {pipeline_path}")
    print(f"  - {labels_path}")
    print(f"  - {metadata_path}")
    print(f"  - {feature_stats_path}")
    print(f"  - {sample_payloads_path}")
    print()
    print(f"Validation accuracy: {accuracy:.4f}")
    print("Top-3 preview from first test sample:")
    print(json.dumps(sample_payloads[0]["top_3_predictions"], indent=2, ensure_ascii=False))


if __name__ == "__main__":
    main()


OSError: dlopen(/Users/apple/.pyenv/versions/3.11.9/lib/python3.11/site-packages/lightgbm/lib/lib_lightgbm.dylib, 0x0006): Library not loaded: @rpath/libomp.dylib
  Referenced from: <D44045CD-B874-3A27-9A61-F131D99AACE4> /Users/apple/.pyenv/versions/3.11.9/lib/python3.11/site-packages/lightgbm/lib/lib_lightgbm.dylib
  Reason: tried: '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/opt/local/lib/libomp/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/local/lib/libomp/libomp.dylib' (no such file), '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/opt/local/lib/libomp/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/local/lib/libomp/libomp.dylib' (no such file), '/Users/apple/.pyenv/versions/3.11.9/lib/libomp.dylib' (no such file)